# singleGBabsorption

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/singleGB/singleGBabsorption.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
# # comment this out if running in colab



In [ ]:
import jax.numpy as jnp
import jax

from beamax import geometry, plotter, utils
from beamax.gb import core, gb_utils, gb_solvers
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions
from beamax.solvers import KWaveSolver


ROOT_DIR = utils.detect_root()
KW_DIR = Path("/path/to/kspaceFirstOrder-OMP")
CACHE_DIR = Path(ROOT_DIR / "cache")
PLOT_DIR = Path(ROOT_DIR / "plots")
CACHE_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update(
    "jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir"
)
pltgb = plotter.PlotHelper()

"""
Plot a Gaussian beam in 1D, 2D, and 3D with absorption.
"""

b = 1
d = 1
N = (2048,) * d
dx = (1 / N[0],) * d


def c(x):
    return 1 + 0 * x[..., 0]


periodic = (True,) * d
cfl = 0.3
domain = geometry.Domain(N=N, dx=dx, c=c, periodic=periodic, cfl=cfl)
XY = domain.grid
ts = domain.generate_time_domain()

domain_size = domain.grid_size

# Gaussian beam parameters
mode = jnp.ones((b,))
x0 = jnp.array([[0.5 * domain_size[0]]])
p0 = jnp.ones((b, d))
p0 = p0.at[:, 1].set(0) if d > 1 else p0
p0 = p0 / jnp.linalg.norm(p0, axis=-1, keepdims=True)
a0 = jnp.ones((b,))

# Beam width parameter
size = 2
alpha0 = jnp.ones((b, d)) * 1j * size / 2
M0 = None
M0 = gb_utils.prepare_M0(alpha0, M0)

lam = 5
c0 = c(jnp.zeros(d))
ω0 = jnp.ones((b,)) * 100

# Solver setup
solver = gb_solvers.solve_ODE_base
solver_config = None

# Compute Gaussian beam for lossless medium
u0_lossless = core.compute_gaussian_beam_real(
    x0,
    p0,
    M0,
    a0,
    ω0,
    mode,
    c,
    0,  # lam=0 for lossless
    ts,
    XY,
    domain_size,
    jnp.array(periodic),
    solver,
    solver_config,
) + core.compute_gaussian_beam_real(
    x0,
    p0,
    M0,
    a0,
    ω0,
    -mode,
    c,
    0,  # lam=0 for lossless
    ts,
    XY,
    domain_size,
    jnp.array(periodic),
    solver,
    solver_config,
)

# Compute Gaussian beam for absorbing medium
u0_absorbing = core.compute_gaussian_beam_real(
    x0,
    p0,
    M0,
    a0,
    ω0,
    mode,
    c,
    lam,  # lam>0 for absorption
    ts,
    XY,
    domain_size,
    jnp.array(periodic),
    solver,
    solver_config,
) + core.compute_gaussian_beam_real(
    x0,
    p0,
    M0,
    a0,
    ω0,
    -mode,
    c,
    lam,  # lam>0 for absorption
    ts,
    XY,
    domain_size,
    jnp.array(periodic),
    solver,
    solver_config,
)

# Squeeze to remove singleton dimensions
ut_lossless = jnp.squeeze(u0_lossless)  # (Nt, N)
ut_absorbing = jnp.squeeze(u0_absorbing)  # (Nt, N)


# Setup K-Wave domains
N_kw = (N[0], 1)
d_kw = len(N_kw)
dx_kw = (dx[0],) * d_kw
periodic_kw = (True,) * d_kw


def convert_lam_to_db(lam, c0):
    """
    Convert wavelength to decibels.
    """
    return float(jnp.log10(jnp.e) / 5 * lam / c0 / 2)


# Lossless domain for K-Wave
domain_kw_lossless = geometry.Domain(
    N=N_kw, dx=dx_kw, c=c, cfl=cfl, periodic=periodic_kw
)

alpha_db = convert_lam_to_db(lam, c0)

print(f"alpha_db: {alpha_db}")

# Absorbing domain for K-Wave
domain_kw_absorbing = geometry.Domain(
    N=N_kw,
    dx=dx_kw,
    c=c,
    cfl=cfl,
    periodic=periodic_kw,
    alpha_power=0,
    alpha_coeff=convert_lam_to_db(lam, c0),
)

binary_mask = jnp.ones(N_kw)

# K-Wave solver setup
simulation_options = SimulationOptions(
    data_cast="single",
    smooth_p0=False,
    save_to_disk=True,
)

execution_options = SimulationExecutionOptions(
    is_gpu_simulation=False,
    delete_data=False,
    verbose_level=0,
    show_sim_log=False,
    binary_path=KW_DIR,
)

kwave_solver = KWaveSolver(simulation_options, execution_options)

# Initial condition (from Gaussian beam at t=0)
p0_init = ut_lossless[0, :, np.newaxis]  # Add singleton dimension for K-Wave

# Run K-Wave simulations
pt_kw_lossless = kwave_solver.forward(p0_init, domain_kw_lossless, binary_mask, ts)
pt_kw_absorbing = kwave_solver.forward(p0_init, domain_kw_absorbing, binary_mask, ts)

In [ ]:
U_loss = np.asarray(ut_lossless)
K_loss = np.asarray(pt_kw_lossless)
U_abs = np.asarray(ut_absorbing)
K_abs = np.asarray(pt_kw_absorbing)

# ============================================================================
# PLOT 1: Initial Condition
# ============================================================================
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)

x_coords = np.array(XY[:, 0])
ax1.plot(x_coords, U_loss[0, :], linewidth=3, color="blue")
ax1.set_title(r"$p_0(x)$", fontsize=16, fontweight="bold")
ax1.set_xlabel(r"$x$", fontsize=14)
# ax1.set_ylabel(r'$p_0$', fontsize=14)
ax1.grid(True, alpha=0.3)

fig1.savefig(
    PLOT_DIR / "initial_condition.png", dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()

# ============================================================================
# PLOT 2: Lossless Comparison
# ============================================================================
fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)

# Shared normalization for first two columns
norm_loss = Normalize(
    vmin=min(U_loss.min(), K_loss.min()), vmax=max(U_loss.max(), K_loss.max())
)

# MSGB Method (Lossless)
im1 = axes2[0].imshow(
    U_loss.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    norm=norm_loss,
    cmap="viridis",
)
axes2[0].set_title(r"$p_t^{\mathrm{MSGB}}(x,t)$", fontsize=12, fontweight="bold")
# axes2[0].set_xlabel(r'$t$', fontsize=12)
# axes2[0].set_ylabel(r'$x$', fontsize=12)
axes2[0].set_xticks([])
axes2[0].set_yticks([])

# K-Wave Method (Lossless)
im2 = axes2[1].imshow(
    K_loss.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    norm=norm_loss,
    cmap="viridis",
)
axes2[1].set_title(r"$p_t^{\text{k-Wave}}(x,t)$", fontsize=12, fontweight="bold")
# axes2[1].set_xlabel(r'$t$', fontsize=12)
# axes2[1].set_ylabel(r'$x$', fontsize=12)
axes2[1].set_xticks([])
axes2[1].set_yticks([])

# Shared colorbar for both methods
cbar1 = fig2.colorbar(
    im1, ax=[axes2[0], axes2[1]], location="right", pad=0.02, shrink=0.9
)
cbar1.set_label(r"$p$", rotation=90, labelpad=15, fontsize=12)

# Difference (Lossless)
D_loss = U_loss - K_loss
m_loss = np.nanmax(np.abs(D_loss))
im3 = axes2[2].imshow(
    D_loss.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vcenter=0.0, vmin=-m_loss, vmax=m_loss),
)
axes2[2].set_title(
    r"$p_t^{\mathrm{MSGB}} - p_t^{\text{k-Wave}}$", fontsize=12, fontweight="bold"
)
# axes2[2].set_xlabel(r'$t$', fontsize=12)
# axes2[2].set_ylabel(r'$x$', fontsize=12)
axes2[2].set_xticks([])
axes2[2].set_yticks([])
cbar3 = fig2.colorbar(im3, ax=axes2[2], shrink=0.8)
cbar3.set_label(r"$\Delta p$", rotation=90, labelpad=15, fontsize=12)

# Maximum amplitude evolution
max_gb_lossless = U_loss.max(axis=1)
max_kw_lossless = K_loss.max(axis=1)
axes2[3].plot(ts, max_gb_lossless, "-", linewidth=2, label="MSGB Method", color="blue")
axes2[3].plot(
    ts, max_kw_lossless, "--", linewidth=2, label="k-Wave Method", color="red"
)
axes2[3].set_title(r"$\max_x |p_t(x)|$", fontsize=12, fontweight="bold")
axes2[3].set_xlabel(r"$t$", fontsize=12)
axes2[3].set_ylabel(r"$\max_x |p_t|$", fontsize=12)
axes2[3].grid(True, alpha=0.3)
axes2[3].legend(fontsize=10)

# fig2.suptitle('Lossless Medium Comparison', fontsize=16, fontweight='bold', y=1.02)
fig2.savefig(
    PLOT_DIR / "lossless_comparison.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

# ============================================================================
# PLOT 3: Absorbing Comparison
# ============================================================================
fig3, axes3 = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)

# Shared normalization for first two columns
norm_abs = Normalize(
    vmin=min(U_abs.min(), K_abs.min()), vmax=max(U_abs.max(), K_abs.max())
)

# MSGB Method (Absorbing)
im4 = axes3[0].imshow(
    U_abs.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    norm=norm_abs,
    cmap="viridis",
)
axes3[0].set_title(r"$p_t^{\mathrm{MSGB}}(x,t)$", fontsize=12, fontweight="bold")
axes3[0].set_xlabel(r"$t$", fontsize=12)
axes3[0].set_ylabel(r"$x$", fontsize=12)
axes3[0].set_xticks([])
axes3[0].set_yticks([])

# K-Wave Method (Absorbing)
im5 = axes3[1].imshow(
    K_abs.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    norm=norm_abs,
    cmap="viridis",
)
axes3[1].set_title(r"$p_t^{\text{k-Wave}}(x,t)$", fontsize=12, fontweight="bold")
# axes3[1].set_xlabel(r'$t$', fontsize=12)
# axes3[1].set_ylabel(r'$x$', fontsize=12)
axes3[1].set_xticks([])
axes3[1].set_yticks([])

# Shared colorbar for both methods
cbar2 = fig3.colorbar(
    im4, ax=[axes3[0], axes3[1]], location="right", pad=0.02, shrink=0.9
)
cbar2.set_label(r"$p$", rotation=90, labelpad=15, fontsize=12)

# Difference (Absorbing)
D_abs = U_abs - K_abs
m_abs = np.nanmax(np.abs(D_abs))
im6 = axes3[2].imshow(
    D_abs.T,
    aspect="auto",
    origin="lower",
    extent=[0, ts[-1], 0, 1],
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vcenter=0.0, vmin=-m_abs, vmax=m_abs),
)
axes3[2].set_title(
    r"$p_t^{\mathrm{MSGB}} - p_t^{\text{k-Wave}}$", fontsize=12, fontweight="bold"
)
# axes3[2].set_xlabel(r'$t$', fontsize=12)
# axes3[2].set_ylabel(r'$x$', fontsize=12)
axes3[2].set_xticks([])
axes3[2].set_yticks([])
cbar6 = fig3.colorbar(im6, ax=axes3[2], shrink=0.8)
cbar6.set_label(r"$\Delta p$", rotation=90, labelpad=15, fontsize=12)

# Maximum amplitude evolution (log scale for absorbing case)
max_gb_abs = U_abs.max(axis=1)
max_kw_abs = K_abs.max(axis=1)
axes3[3].plot(ts, max_gb_abs, "-", linewidth=2, label="MSGB Method", color="blue")
axes3[3].plot(ts, max_kw_abs, "--", linewidth=2, label="k-Wave Method", color="red")
axes3[3].set_title(r"$\max_x |p_t(x)|$", fontsize=12, fontweight="bold")
axes3[3].set_xlabel(r"$t$", fontsize=12)
axes3[3].set_ylabel(r"$\max_x |p_t|$", fontsize=12)
axes3[3].set_yscale("log")
axes3[3].grid(True, alpha=0.3)
axes3[3].legend(fontsize=10)

# fig3.suptitle('Absorbing Medium Comparison', fontsize=16, fontweight='bold', y=1.02)
fig3.savefig(
    PLOT_DIR / "absorbing_comparison.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()